In [1]:
# from sklearn.datasets import fetch_20newsgroups
# from sklearn.model_selection import train_test_split

# news_data = fetch_20newsgroups(subset='all', remove=tuple(['headers', 'footers', 'quotes']), random_state=42)
# print(type(news_data))
# text_train, text_test, label_train, label_test = train_test_split(news_data.data, news_data.target, test_size=0.2, random_state=42)

In [2]:
# import re

# def clean_text(text):
#     # 1. 소문자화
#     text = text.lower()
#     # 2. 특수문자 제거 (단어가 아닌 문자 제거)
#     text = re.sub(r'[^a-z\s]', '', text)
#     # 3. 여러 공백을 하나로
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# # 전체 데이터에 적용
# text_train_cleaned = [clean_text(t) for t in text_train]
# text_test_cleaned = [clean_text(t) for t in text_test]

In [3]:
# text_train_cleaned[0]

In [4]:
import pandas as pd

# text_train_df = pd.DataFrame({'text': text_train_cleaned, 'label': label_train})
# text_test_df = pd.DataFrame({'text': text_test_cleaned, 'label': label_test})

text_train_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\data\text_train.csv"
text_test_path = r"C:\Users\user\Desktop\PythonWorkspace\Codeit_Sprint_AI01\16\data\text_test.csv"
text_train_df = pd.read_csv(text_train_path, keep_default_na=False)
text_test_df = pd.read_csv(text_test_path, keep_default_na=False)

text_train_df.head()

,text,label
0,ive gotten very few posts on this group in the...,5
1,interesting id fight the ticket first off ther...,8
2,i remember as a kid visiting my relatives on k...,13
3,it can be painless so it isnt cruel and it has...,0
4,the owners are whining about baseball not bein...,9


In [5]:
# # 추론을 위해서 데이터 저장

# text_train_df.to_csv("text_train.csv", index=False)
# text_test_df.to_csv("text_test.csv", index=False)

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased", use_fast=True)

class NewsDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]['text']
        label = self.data.iloc[idx]['label']

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        label = torch.tensor(label, dtype=torch.long)
        return input_ids, attention_mask, label

train_dataset = NewsDataset(text_train_df, tokenizer)
test_dataset = NewsDataset(text_test_df, tokenizer)

In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
import torch
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, max_len=256, num_classes=20, num_layers=3, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)

        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4, dropout=0.1, batch_first=True)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.size()
        positions = torch.arange(0, T, device=input_ids.device).unsqueeze(0).expand(B, T)
        x = self.embedding(input_ids) + self.pos_embedding(positions)  # [B, T, E]

        # padding mask: True는 마스킹됨
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None

        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)

        x = self.norm(x)
        pooled = x.mean(dim=1)  # 평균 풀링
        return self.classifier(pooled)

In [22]:
from tqdm import tqdm

def train_model(model, train_loader, test_loader, optimizer, criterion, device, num_epochs=10, patience=3):
    model.to(device)

    best_val = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for input_ids, attention_mask, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * input_ids.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        avg_train_loss = total_loss / total
        train_acc = correct / total
        print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f}, Accuracy: {train_acc:.4f}")

        val_loss, val_acc = evaluate_model(model, test_loader, criterion, device)
        print(f"[Epoch {epoch+1}] Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

        # EarlyStopping check
        if val_acc > best_val:
            best_val = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "nlp_weight.pt")
            print("→ Validation loss improved. Saving model.")
        else:
            patience_counter += 1
            print(f"→ No improvement. EarlyStopping patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("→ Early stopping triggered.")
                model.load_state_dict(torch.load("nlp_weight.pt", map_location=device))
                break

def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(data_loader, leave=False):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * input_ids.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    print(f"[Validation/Test] Loss: {avg_loss:.4f}, Accuracy: {acc:.4f}")
    return avg_loss, acc

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleClassifier(vocab_size=tokenizer.vocab_size, num_classes=20)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [20]:
train_model(model, train_loader, test_loader, optimizer, criterion, device, num_epochs=100, patience=5)

[Epoch 1] Train Loss: 2.9723, Accuracy: 0.0724


[Epoch 1] Val Loss: 2.9006, Accuracy: 0.1040
→ Validation loss improved. Saving model.


[Epoch 2] Train Loss: 2.7992, Accuracy: 0.1142


[Epoch 2] Val Loss: 2.6658, Accuracy: 0.1462
→ Validation loss improved. Saving model.


[Epoch 3] Train Loss: 2.5839, Accuracy: 0.1773


...

[Epoch 21] Train Loss: 0.8269, Accuracy: 0.7212


[Epoch 21] Val Loss: 2.3861, Accuracy: 0.4308
→ No improvement. EarlyStopping patience: 2/5


[Epoch 22] Train Loss: 0.7691, Accuracy: 0.7412


[Epoch 22] Val Loss: 2.4932, Accuracy: 0.4300
→ No improvement. EarlyStopping patience: 3/5


KeyboardInterrupt: 

In [23]:
model.load_state_dict(torch.load("nlp_weight.pt", map_location=device))
test_loss, test_acc = evaluate_model(model, test_loader, criterion, device)

  0%|          | 0/118 [00:00<?, ?it/s]

[Validation/Test] Loss: 2.2646, Accuracy: 0.4334


In [24]:
model.eval()

quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

# 저장
torch.save(quantized_model, "nlp_quantized.pt")

In [27]:
import torch.nn.utils.prune as prune

pruned_model = SimpleClassifier(vocab_size=tokenizer.vocab_size)
pruned_model.load_state_dict(torch.load("nlp_weight.pt", map_location="cpu"))

# 프루닝 대상 지정: classifier 내부 Linear 계층 2개
modules_to_prune = [
    (pruned_model.classifier[0], 'weight'),  # Linear(embed_dim → 128)
    (pruned_model.classifier[3], 'weight'),  # Linear(128 → num_classes)
]

# 전체 weight의 30%를 L1 기준으로 제거
prune.global_unstructured(
    modules_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.3
)

# pruning mask 제거 → 실제 weight에 반영
for module, name in modules_to_prune:
    prune.remove(module, name)

# 저장
torch.save(pruned_model.state_dict(), "nlp_pruned.pt")

In [26]:
import torch.onnx

# 모델을 CPU로 옮김
model = model.to("cpu")
model.eval()

# 입력도 CPU에 생성
dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 128)).to("cpu")
dummy_attention_mask = torch.ones(1, 128, dtype=torch.long).to("cpu")

# ONNX Export
torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask),
    "nlp_model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14,
    do_constant_folding=True
)

In [28]:
import os

for f in os.listdir("."):
    if os.path.isfile(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"{f}: {size_mb:.2f} MB")

bert_modeling.ipynb: 0.04 MB
mnist_inference.ipynb: 0.07 MB
mnist_modeling.ipynb: 0.20 MB
nlp_inference.ipynb: 0.03 MB
nlp_model.onnx: 17.46 MB
nlp_modeling.ipynb: 0.10 MB
nlp_pruned.pt: 17.39 MB
nlp_quantized.pt: 16.22 MB
nlp_weight.pt: 17.39 MB
README.md: 0.00 MB


In [29]:
!pip install torchinfo

In [30]:
from torchinfo import summary

dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 128)).to("cpu")
dummy_attention_mask = torch.ones(1, 128, dtype=torch.long).to("cpu")

summary(model, input_data=(dummy_input_ids, dummy_attention_mask), device="cpu")

Layer (type:depth-idx)                   Output Shape              Param #
SimpleClassifier                         [1, 20]                   --
├─Embedding: 1-1                         [1, 128, 128]             3,906,816
├─Embedding: 1-2                         [1, 128, 128]             32,768
├─ModuleList: 1-3                        --                        --
│    └─TransformerEncoderLayer: 2-1      [1, 128, 128]             --
│    │    └─MultiheadAttention: 3-1      [1, 128, 128]             66,048
│    │    └─Dropout: 3-2                 [1, 128, 128]             --
│    │    └─LayerNorm: 3-3               [1, 128, 128]             256
│    │    └─Linear: 3-4                  [1, 128, 512]             66,048
│    │    └─Dropout: 3-5                 [1, 128, 512]             --
│    │    └─Linear: 3-6                  [1, 128, 128]             65,664
│    │    └─Dropout: 3-7                 [1, 128, 128]             --
│    │    └─LayerNorm: 3-8               [1, 128, 128]       